# Route Factory

> Route factory for job trigger, progress polling, and cancellation.

In [ ]:
#| default_exp routes.init

In [ ]:
#| export
from typing import Any, Callable, Dict, List, Optional, Tuple
from dataclasses import asdict

from fasthtml.common import Div, Span, Script, APIRouter, FT

from cjm_fasthtml_interactions.core.state_store import get_session_id
from cjm_workflow_state.state_store import SQLiteWorkflowStateStore

from cjm_fasthtml_job_monitor.html_ids import JobMonitorHtmlIds
from cjm_fasthtml_job_monitor.models import JobMonitorUrls, JobMonitorConfig, ResourceSnapshot
from cjm_fasthtml_job_monitor.services.monitor import JobMonitorService
from cjm_fasthtml_job_monitor.components.trigger import render_job_trigger, render_job_progress_button
from cjm_fasthtml_job_monitor.components.overlay import render_job_overlay, render_job_overlay_placeholder
from cjm_fasthtml_job_monitor.components.modal import render_job_modal, render_poll_response

## init_job_monitor_routes

Factory function that creates the three routes (trigger, progress, cancel)
and returns them as an `APIRouter` plus a `JobMonitorUrls` for consumers to reference.

### Consumer Setup

The consumer must:
1. Place `render_job_trigger(config, ids, urls)` in their toolbar/action area
2. Place `render_job_overlay_placeholder(ids)` inside the content area to overlay
   (the content area container must have `position: relative`)
3. Place `render_job_modal(...)` at page level (or let the trigger route create it)
4. Provide callbacks for job lifecycle events

In [ ]:
#| export
def _get_job_data(service, job_id):
    """Extract job status fields from a Job object."""
    job = service.get_job(job_id)
    if not job:
        return None, 'pending', 0.0, '', None, None
    return (
        job,
        job.status.value if hasattr(job.status, 'value') else str(job.status),
        job.progress or 0.0,
        job.status_message or '',
        job.started_at,
        job.completed_at,
    )

In [ ]:
#| export
def init_job_monitor_routes(
    monitor_service: JobMonitorService,           # Service instance
    plugin_name: str,                             # Target plugin for jobs
    state_store: SQLiteWorkflowStateStore,         # For persisting job_id
    workflow_id: str,                             # Workflow ID for state access
    step_id: str,                                 # Step ID for state access
    state_key: str,                               # Key in step state for job_id
    prefix: str,                                  # URL prefix
    overlay_target_id: str,                       # ID of element to overlay
    kb_system_id: Optional[str] = None,           # Keyboard system ID to pause/resume
    on_complete: Optional[Callable] = None,       # async fn(job, request, sess) -> List[FT]
    on_cancel: Optional[Callable] = None,         # async fn(job, request, sess) -> List[FT]
    on_fail: Optional[Callable] = None,           # async fn(job, request, sess) -> List[FT]
    job_args_builder: Optional[Callable] = None,  # fn(state_store, workflow_id, session_id) -> (args, kwargs)
    config: Optional[JobMonitorConfig] = None,    # UI config
    id_prefix: str = "jm",                        # HTML ID prefix
    icon_fn: Optional[Callable] = None,           # Icon renderer fn(name, **kwargs) -> FT
) -> Tuple[APIRouter, JobMonitorUrls, JobMonitorHtmlIds]:  # (router, urls, ids)
    """Initialize job monitor routes."""
    config = config or JobMonitorConfig()
    ids = JobMonitorHtmlIds(prefix=id_prefix)

    router = APIRouter(prefix=prefix)

    @router
    async def trigger(request, sess):
        """Submit a job and return the modal + overlay + progress button via OOB."""
        session_id = get_session_id(sess)

        # Build job args from consumer callback
        args, kwargs = (), {}
        if job_args_builder:
            args, kwargs = job_args_builder(state_store, workflow_id, session_id)

        # Submit to queue
        job_id = await monitor_service.submit_job(plugin_name, *args, **kwargs)

        # Persist job_id in step state
        state = state_store.get_state(workflow_id, session_id)
        step_states = state.get("step_states", {})
        step_state = step_states.get(step_id, {})
        step_state[state_key] = job_id
        step_states[step_id] = step_state
        state["step_states"] = step_states
        state_store.update_state(workflow_id, session_id, state)

        # Build URLs for progress/cancel
        urls = JobMonitorUrls(
            trigger=trigger.to(),
            progress=progress.to(),
            cancel=cancel.to(),
        )

        # Get initial job data
        _, status, prog_val, msg, started, completed = _get_job_data(monitor_service, job_id)
        logs = monitor_service.get_logs(plugin_name, config.log_lines)
        resources = monitor_service.get_resource_snapshot(plugin_name)

        # Build OOB responses
        # NOTE: Use hyphenated 'hx-swap-oob' when assigning to .attrs dict
        oob_parts = []

        # 1. Progress button replaces trigger slot (has id=ids.trigger_slot)
        prog_btn = render_job_progress_button(config, ids)
        prog_btn.attrs['hx-swap-oob'] = "true"
        oob_parts.append(prog_btn)

        # 2. Overlay replaces overlay placeholder (has id=ids.overlay)
        overlay = render_job_overlay(ids, config)
        overlay.attrs['hx-swap-oob'] = "true"
        oob_parts.append(overlay)

        # 3. Modal replaces modal placeholder (has id=ids.modal)
        modal_el = render_job_modal(
            config, ids, urls, status=status, progress_value=prog_val,
            status_message=msg, started_at=started, completed_at=completed,
            logs=logs, resources=resources, open_on_render=True,
        )
        modal_el.attrs['hx-swap-oob'] = "true"
        oob_parts.append(modal_el)

        # 4. Keyboard pause script
        if kb_system_id:
            oob_parts.append(Script(f"window.kbCoordinator?.pause('{kb_system_id}');"))

        return tuple(oob_parts)

    @router
    async def progress(request, sess):
        """Poll job progress — return updated poll anchor + OOB tab content."""
        session_id = get_session_id(sess)

        # Build URLs
        urls = JobMonitorUrls(
            trigger=trigger.to(),
            progress=progress.to(),
            cancel=cancel.to(),
        )

        # Get job_id from state
        state = state_store.get_state(workflow_id, session_id)
        step_states = state.get("step_states", {})
        step_state = step_states.get(step_id, {})
        job_id = step_state.get(state_key)

        if not job_id:
            # No active job — return inert poll anchor
            return render_poll_response(config, ids, urls)

        # Get current job state
        job, status, prog_val, msg, started, completed = _get_job_data(monitor_service, job_id)
        if not job:
            return render_poll_response(config, ids, urls)

        # Get logs and resources
        logs = monitor_service.get_logs(plugin_name, config.log_lines)
        resources = monitor_service.get_resource_snapshot(plugin_name)

        # Check for terminal state
        terminal_statuses = {'completed', 'failed', 'cancelled'}
        if status in terminal_statuses:
            # Clear job_id from state
            step_state.pop(state_key, None)
            step_states[step_id] = step_state
            state["step_states"] = step_states
            state_store.update_state(workflow_id, session_id, state)

            # Fire callback
            extra_oob = []
            if status == 'completed' and on_complete:
                result = await on_complete(job, request, sess)
                if result:
                    extra_oob.extend(result if isinstance(result, (list, tuple)) else [result])
            elif status == 'cancelled' and on_cancel:
                result = await on_cancel(job, request, sess)
                if result:
                    extra_oob.extend(result if isinstance(result, (list, tuple)) else [result])
            elif status == 'failed' and on_fail:
                result = await on_fail(job, request, sess)
                if result:
                    extra_oob.extend(result if isinstance(result, (list, tuple)) else [result])

            # Cleanup OOB: remove overlay (replace with empty placeholder)
            overlay_placeholder = render_job_overlay_placeholder(ids)
            overlay_placeholder.attrs['hx-swap-oob'] = "true"
            extra_oob.append(overlay_placeholder)

            # Restore trigger button
            trigger_btn = render_job_trigger(config, ids, urls, icon_fn=icon_fn)
            trigger_btn.attrs['hx-swap-oob'] = "true"
            extra_oob.append(trigger_btn)

            # Resume keyboard
            if kb_system_id:
                extra_oob.append(Script(f"window.kbCoordinator?.resume('{kb_system_id}');"))

            # Poll response (anchor stops polling) + tab updates + extra OOB
            poll_parts = render_poll_response(
                config, ids, urls, status=status, progress_value=prog_val,
                status_message=msg, started_at=started, completed_at=completed,
                logs=logs, resources=resources,
            )
            return (*poll_parts, *extra_oob)

        # Still running — return poll response (anchor continues polling) + tab updates
        return render_poll_response(
            config, ids, urls, status=status, progress_value=prog_val,
            status_message=msg, started_at=started, completed_at=completed,
            logs=logs, resources=resources,
        )

    @router
    async def cancel(request, sess):
        """Cancel the active job."""
        session_id = get_session_id(sess)

        state = state_store.get_state(workflow_id, session_id)
        step_states = state.get("step_states", {})
        step_state = step_states.get(step_id, {})
        job_id = step_state.get(state_key)

        if job_id:
            await monitor_service.cancel_job(job_id)

        # Cancellation is async — the next progress poll detects the terminal state
        return ""

    # Build URLs
    urls = JobMonitorUrls(
        trigger=trigger.to(),
        progress=progress.to(),
        cancel=cancel.to(),
    )

    return router, urls, ids

## Resume Helper

Utility for consumers to check for in-flight jobs on page load.

In [ ]:
#| export
def check_inflight_job(
    monitor_service: JobMonitorService,       # Service instance
    plugin_name: str,                         # Target plugin
    state_store: SQLiteWorkflowStateStore,    # State store
    workflow_id: str,                         # Workflow ID
    session_id: str,                          # Session ID
    step_id: str,                             # Step ID
    state_key: str,                           # State key for job_id
    config: JobMonitorConfig,                 # Display config
    ids: JobMonitorHtmlIds,                   # Element IDs
    urls: JobMonitorUrls,                     # Route URLs
    icon_fn: Optional[Callable] = None,       # Icon renderer
) -> Tuple[Optional[FT], Optional[FT], bool]:  # (trigger_or_progress_btn, overlay_or_placeholder, is_running)
    """Check for in-flight job and return appropriate UI state.
    
    Returns:
        - Button element (trigger or progress button)
        - Overlay element (active overlay or empty placeholder)
        - Whether a job is currently running
    """
    state = state_store.get_state(workflow_id, session_id)
    step_states = state.get("step_states", {})
    step_state = step_states.get(step_id, {})
    job_id = step_state.get(state_key)

    if not job_id:
        # No in-flight job
        return (
            render_job_trigger(config, ids, urls, icon_fn=icon_fn),
            render_job_overlay_placeholder(ids),
            False,
        )

    # Check job status
    job, status, _, _, _, _ = _get_job_data(monitor_service, job_id)

    if not job or status in {'completed', 'failed', 'cancelled'}:
        # Job finished -- clean up state
        step_state.pop(state_key, None)
        step_states[step_id] = step_state
        state["step_states"] = step_states
        state_store.update_state(workflow_id, session_id, state)

        return (
            render_job_trigger(config, ids, urls, icon_fn=icon_fn),
            render_job_overlay_placeholder(ids),
            False,
        )

    # Job is still running
    return (
        render_job_progress_button(config, ids),
        render_job_overlay(ids, config),
        True,
    )

In [ ]:
# Test URL generation
from cjm_fasthtml_job_monitor.models import JobMonitorUrls

# Verify URLs are constructed correctly
urls = JobMonitorUrls(
    trigger="/workflow/seg/fa/trigger",
    progress="/workflow/seg/fa/progress",
    cancel="/workflow/seg/fa/cancel",
)
assert urls.trigger.endswith("/trigger")
assert urls.progress.endswith("/progress")
assert urls.cancel.endswith("/cancel")
print(f"URLs: {urls}")
print("Route factory URL construction: OK")

URLs: JobMonitorUrls(trigger='/workflow/seg/fa/trigger', progress='/workflow/seg/fa/progress', cancel='/workflow/seg/fa/cancel')
Route factory URL construction: OK


In [ ]:
# Test check_inflight_job with no active job
from cjm_fasthtml_job_monitor.html_ids import JobMonitorHtmlIds
from cjm_fasthtml_job_monitor.models import JobMonitorConfig

class MockStateStore:
    def __init__(self, state=None):
        self._state = state or {}
    def get_state(self, wf_id, sess_id):
        return dict(self._state)
    def update_state(self, wf_id, sess_id, state):
        self._state = state

ids = JobMonitorHtmlIds(prefix="jm")
config = JobMonitorConfig()
urls = JobMonitorUrls(trigger="/trigger", progress="/progress", cancel="/cancel")
store = MockStateStore({"step_states": {"seg": {}}})

btn_el, overlay_el, is_running = check_inflight_job(
    monitor_service=None,  # Not needed when no job_id in state
    plugin_name="test-plugin",
    state_store=store,
    workflow_id="wf1",
    session_id="s1",
    step_id="seg",
    state_key="fa_job_id",
    config=config,
    ids=ids,
    urls=urls,
)
assert not is_running
assert btn_el.attrs['id'] == 'jm-trigger-slot'
assert overlay_el.attrs['id'] == 'jm-overlay'
assert len(overlay_el.children) == 0  # Placeholder
print("check_inflight_job (no active job): OK")

check_inflight_job (no active job): OK


In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()